# Plotting pileups for PcG RCMC

## Imports

In [1]:
import os
import numpy as np

from matplotlib import pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from coolpuppy import coolpup
from coolpuppy.lib import numutils
from coolpuppy.lib.puputils import divide_pups
from coolpuppy.lib.puputils import get_score
from coolpuppy import plotpup
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooler
import bioframe
import cooltools
from cooltools import expected_cis, expected_trans
from cooltools.lib import plotting
import cooler as clr
import h5py
import pickle
import math

from matplotlib.gridspec import GridSpec

import itertools

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd

In [2]:
#Things I added
import glob
from pathlib import Path

In [3]:
plt.rcParams.update({"text.usetex": False})
plt.rc('pdf',fonttype = 42)

In [90]:
#Function to get normalised mean diagonals to plot profiles
def getMeanDiagNorm(mcool, max_offset, norm_shoulder):
    diags = []
    for i in range(-max_offset, max_offset+1):
        diagonal = np.diagonal(mcool['data'], offset = i)
        extra_length = max_offset - abs(i)
        startpos = math.ceil(extra_length/2)
        endpos = -math.floor(extra_length/2)
        if endpos == 0:
            diagonal_reduced = diagonal[startpos:]
        else:
            diagonal_reduced = diagonal[startpos:endpos]
        diags.append(diagonal_reduced)
    diags_array = np.stack(diags)
    mean_diags = diags_array.mean(axis = 0)
    #Then normalise the array based on the first x values from both sides
    left_side = mean_diags[0:norm_shoulder]
    right_side = mean_diags[-norm_shoulder:]
    both_sides = np.concatenate((left_side, right_side), axis = 0)
    norm_val = both_sides.mean()
    norm_diags = mean_diags/norm_val
    
    
    return norm_diags

## Read in data

### Get mcools

In [ ]:
mcool_path = "/path/to/directory/of/mcools/"
 
conditions = sorted(glob.glob(f'{mcool_path}batch?/*.mcool', recursive=True))
#Also get the names:
conditionNames = [Path(mcool).stem.split('_merged')[0] for mcool in conditions]
conditionNames


In [127]:
resolution = 5000

In [128]:
expected_path = "/path/to/save/expecteds"
Path(expected_path).mkdir(parents=True, exist_ok=True)

### Import interactions

In [129]:
PcG_sites = pd.read_csv('/path/to/interaction.bedpe', sep='\t')


In [130]:
loop_dict = dict(((k, eval(k)) for k in ('PcG_sites')))

In [ ]:
loop_dfs = []
for name, loopset in loop_dict.items():
    loopset['loopname'] = name
    loop_dfs.append(loopset)
    
all_loops = pd.concat(loop_dfs)
all_loops

### Compute expected

In [ ]:
# Use bioframe to fetch the genomic features from the UCSC (NB I think in newer versions of bioframe the first column is called 'name' by default, not 'index')
mm39_chromsizes = bioframe.fetch_chromsizes('mm39')
df_mm39 = pd.DataFrame.from_dict(mm39_chromsizes).reset_index(level=0)
df_mm39 = df_mm39.rename(columns={"index": "chrom", "length": "end"})
df_mm39['start'] = 0
df_mm39['chrom'] = df_mm39['name']
df_mm39 = df_mm39[['chrom', 'start', 'end', 'name']]
df_mm39

In [ ]:
# Compute expected
expected_df_list = []
output_names = [f"{expected_path}{name}_expected_cis_{resolution}.tsv" for name in conditionNames]
for mcool_file, output_name in zip(conditions, output_names):
    curr_cooler = cooler.Cooler(f"{mcool_file}::/resolutions/{resolution}")
    expected = expected_cis(
        curr_cooler,
        ignore_diags=0,
        view_df=df_mm39,
        chunksize=1000000,
        nproc = 32)
    expected_df_list.append(expected)
    expected.to_csv(output_name, sep='\t', index=False, header=True) # Let's save to use in CLI

## Make pileups for all loop combinations

In [ ]:
flanksize = 50000

colormap = 'coolwarm'
pileup_dfs = []

for cool, name, expect in zip(conditions, conditionNames, expected_df_list):
    print(name)
    curr_cooler = cooler.Cooler(f'{cool}::/resolutions/{resolution}')
    curr_bedpe = all_loops
    curr_pileup = coolpup.pileup(curr_cooler, curr_bedpe, "bedpe", view_df=df_mm39, expected_df=expect, rescale=True, rescale_size=99, rescale_flank = 3, nproc = 24, groupby = ['loopname'])
    
    pileup_dfs.append(curr_pileup)
    
all_pileups_loopstpyes = pd.concat(pileup_dfs)


### Plot and output

In [ ]:
colormap = 'coolwarm'

fg = plotpup.plot(all_pileups_loopstpyes, cols='loopname', rows = 'cooler',
                  score=True, cmap=colormap, scale='log', #sym=False,
                  height=5, despine=False, vmax=2.5, center = 10)

plot_dir = "/mnt/coldstorage/shares/Miles/Analysis/Polycomb/RCMC/pileups/" # output path for plots
name = f"PcG_pileups_allloops_{resolution}res_{flanksize}flank_{colormap}_all.pdf" # name for plot output
plt.savefig(os.path.join(plot_dir, name), format='pdf')

## Plot 1D plots over pileup diagonal

In [355]:
filtered_pileups = all_pileups_loopstpyes[all_pileups_loopstpyes['loopname'] == "PcG_sites"]
filtered_pileups['name'] = conditionNames

In [356]:
max_offset = 10
norm_shoulder = 10

In [ ]:
#Manually adjust dimensions of grid to make sure all are plotted
fig = plt.figure(figsize = (10, 20))
gs = GridSpec(13, 1)

plotcount = 0
for rownum, mcool in filtered_pileups.iterrows():
    if plotcount == 0:
        ax1 = fig.add_subplot(gs[plotcount])
        ax1.plot(getMeanDiagNorm(mcool, max_offset, norm_shoulder), label = mcool['name'])
    elif plotcount == 1:
        ax1.plot(getMeanDiagNorm(mcool, max_offset, norm_shoulder), label = mcool['name'])
        ax1.legend(bbox_to_anchor=(1.05, 1), loc = "upper left", borderaxespad=0.)
        ax1.set_box_aspect(1)
    elif plotcount % 2 == 0:
        ax = fig.add_subplot(gs[plotcount//2], sharey = ax1)
        ax.plot(getMeanDiagNorm(mcool, max_offset, norm_shoulder), label = mcool['name'])
    else:
        ax.plot(getMeanDiagNorm(mcool, max_offset, norm_shoulder), label = mcool['name'])
        ax.legend(bbox_to_anchor=(1.05, 1), loc = "upper left", borderaxespad=0.)
        ax.set_box_aspect(1)
    plotcount += 1
plot_dir = "/path/for/output/" # output path for plots
name = f"PcG_pileups_{resolution}res_{flanksize}flank_{colormap}_1D.svg" # name for plot output
plt.savefig(os.path.join(plot_dir, name), format='pdf')